# 06 - t=-2 pooled and conference breakdown figures

This notebook creates compact t=-2 reference event study figures that
show the pooled estimate and the conference breakdown using the same
sample definition in every panel.

It also creates a t=-2 citation count companion to the citation source
shift figure. The source shift check uses a narrower filter
because it keeps only event units with exactly one observed
same conference PC service year. The final table in this notebook makes
that filter difference explicit.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import IFrame, Markdown, display
from matplotlib import font_manager

from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_4_PREPARED = PROJECT / "step_4_data" / "prepared"
STEP_4_SUMMARY = PROJECT / "step_4_artifacts" / "summary_tables"
T_MINUS_FIGURES = PROJECT / "step_4_artifacts" / "figures_t_minus_event_study"
REPORT_FIGURES = PROJECT / "report_latex" / "figures"
ensure_dirs(STEP_4_SUMMARY, T_MINUS_FIGURES, REPORT_FIGURES)

T_MINUS_ROWS_PATH = STEP_4_PREPARED / "t_minus_2_reference_event_window_rows.parquet"
SOURCE_SHIFT_ROWS_PATH = STEP_4_PREPARED / "citation_source_shift_event_rows.parquet"

T_MINUS_MAIN_FIGURE_OUT = T_MINUS_FIGURES / "t_minus_no_earlier_broad_service_main_text.pdf"
T_MINUS_MAIN_RAW_FIGURE_OUT = (
    T_MINUS_FIGURES / "t_minus_no_earlier_broad_service_main_text_raw.pdf"
)
REPORT_T_MINUS_MAIN_FIGURE_OUT = (
    REPORT_FIGURES / "t_minus_no_earlier_broad_service_main_text.pdf"
)
REPORT_T_MINUS_MAIN_RAW_FIGURE_OUT = (
    REPORT_FIGURES / "t_minus_no_earlier_broad_service_main_text_raw.pdf"
)

T_MINUS_SOURCE_SHIFT_COMPANION_FIGURE_OUT = (
    T_MINUS_FIGURES / "t_minus_source_shift_companion.pdf"
)
REPORT_T_MINUS_SOURCE_SHIFT_COMPANION_FIGURE_OUT = (
    REPORT_FIGURES / "t_minus_source_shift_companion.pdf"
)

FILTER_COMPARISON_OUT = STEP_4_SUMMARY / "tminus_source_filter_check.csv"

EVENT_TIMES = np.array([-2, -1, 0, 1, 2])
BOOTSTRAP_REPS = 2000

print("Project folder: .")
print(f"Run mode: {setup.run_mode}")
print(f"Overwrite artifacts: {setup.overwrite_artifacts}")

Project folder: .
Run mode: fast
Overwrite artifacts: True


## 2. Plot style

In [2]:
font_path = PROJECT / "fonts" / "LinLibertine_R.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    font_prop = font_manager.FontProperties(fname=font_path)
    font_family = font_prop.get_name()
else:
    font_family = "serif"

mpl.rcParams.update(
    {
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "font.size": 9,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "font.family": font_family,
        "text.usetex": True,
        "axes.linewidth": 0.75,
        "axes.edgecolor": "black",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

CONFERENCE_COLORS = {
    "All": "#555555",
    "ICFP": "#f5bde6",
    "POPL": "#81c8be",
    "OOPSLA": "#8aadf4",
    "PLDI": "#e5c890",
    "PLDI 2021 onward": "#e5c890",
}

## 3. Load t=-2 event window rows

In [3]:
event_rows = pd.read_parquet(T_MINUS_ROWS_PATH)
event_rows["event_time"] = event_rows["event_time"].astype(int)
event_rows["conference_group"] = event_rows["conference"].replace(
    {"OOPSLA1": "OOPSLA", "OOPSLA2": "OOPSLA"}
)
aggregate_groups = ["ICFP", "POPL", "PLDI 2021 onward"]

required_columns = {
    "event_unit_id",
    "conference",
    "conference_group",
    "plot_group",
    "event_time",
    "delta_from_t_minus_2_log10_citations",
    "is_true_first_broad_service_in_observed_year_conf",
    "n_pc_years_this_conference",
}
missing = required_columns - set(event_rows.columns)
assert not missing, missing

unit_rows = event_rows.drop_duplicates("event_unit_id")
if "delta_from_t_minus_2_raw_citations" not in event_rows.columns:
    reference_raw = (
        event_rows.loc[
            event_rows["event_time"].eq(-2),
            ["event_unit_id", "citation_count"],
        ].rename(columns={"citation_count": "reference_raw_citations_t_minus_2"})
    )
    event_rows = event_rows.merge(
        reference_raw,
        on="event_unit_id",
        how="left",
        validate="many_to_one",
    )
    event_rows["delta_from_t_minus_2_raw_citations"] = (
        event_rows["citation_count"] - event_rows["reference_raw_citations_t_minus_2"]
    )
check = event_rows.loc[event_rows["event_time"].eq(-2)]
assert np.allclose(check["delta_from_t_minus_2_raw_citations"].fillna(0), 0)

print(f"t=-2 rows: {len(event_rows):,}")
print(f"event units: {unit_rows['event_unit_id'].nunique():,}")
display(unit_rows["conference_group"].value_counts().rename("event_units"))

t=-2 rows: 2,015
event units: 403


conference_group
PLDI      138
POPL      138
ICFP      106
OOPSLA     21
Name: event_units, dtype: int64

## 4. Helpers

In [4]:
def event_units(frame):
    return frame.sort_values(["event_unit_id", "event_time"]).drop_duplicates(
        "event_unit_id"
    )


def no_earlier_broad_service(frame):
    return frame.loc[
        frame["is_true_first_broad_service_in_observed_year_conf"].eq(True)
    ].copy()


def bootstrap_summary(frame, value_col, seed=1234):
    if frame.empty:
        return pd.DataFrame(
            columns=["event_time", "mean", "ci_lower", "ci_upper", "n_event_units"]
        )

    matrix = (
        frame.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values=value_col,
            aggfunc="mean",
        )
        .reindex(columns=EVENT_TIMES)
        .to_numpy()
    )
    n_units = matrix.shape[0]
    if n_units == 0:
        return pd.DataFrame(
            columns=["event_time", "mean", "ci_lower", "ci_upper", "n_event_units"]
        )

    means = np.nanmean(matrix, axis=0)
    rng = np.random.default_rng(seed)
    boot = np.empty((BOOTSTRAP_REPS, len(EVENT_TIMES)))
    for draw in range(BOOTSTRAP_REPS):
        idx = rng.choice(n_units, size=n_units, replace=True)
        boot[draw] = np.nanmean(matrix[idx], axis=0)

    return pd.DataFrame(
        {
            "event_time": EVENT_TIMES,
            "mean": means,
            "ci_lower": np.nanpercentile(boot, 2.5, axis=0),
            "ci_upper": np.nanpercentile(boot, 97.5, axis=0),
            "n_event_units": n_units,
        }
    )


def draw_event_axis(ax, frame, color, title, seed, show_ylabel=True):
    summary = bootstrap_summary(
        frame,
        "delta_from_t_minus_2_log10_citations",
        seed=seed,
    )
    ax.axhline(0, color="0.78", linewidth=0.45, zorder=0)
    ax.axvline(0, color="black", linewidth=0.75, linestyle="--")

    if not summary.empty:
        ax.scatter(
            summary["event_time"],
            summary["mean"],
            s=22,
            color=color,
            zorder=3,
        )
        ci_rows = summary.loc[summary["event_time"].ne(-2)].copy()
        if not ci_rows.empty:
            yerr = np.vstack(
                [
                    ci_rows["mean"].to_numpy() - ci_rows["ci_lower"].to_numpy(),
                    ci_rows["ci_upper"].to_numpy() - ci_rows["mean"].to_numpy(),
                ]
            )
            ax.errorbar(
                ci_rows["event_time"],
                ci_rows["mean"],
                yerr=yerr,
                fmt="none",
                ecolor="black",
                elinewidth=0.8,
                capsize=2.7,
                capthick=0.8,
                zorder=4,
            )

    ax.set_title(title, loc="left", pad=4)
    ax.set_xticks(EVENT_TIMES)
    ax.set_xlabel("Event time")
    if show_ylabel:
        ax.set_ylabel(r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$")
    else:
        ax.set_ylabel("")
    ax.grid(axis="y", color="0.88", linestyle=":", linewidth=0.55)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return summary


def apply_shared_ylim(axes, summaries):
    values = []
    for summary in summaries:
        if summary.empty:
            continue
        values.extend(summary["mean"].dropna().tolist())
        values.extend(summary["ci_lower"].dropna().tolist())
        values.extend(summary["ci_upper"].dropna().tolist())
    if not values:
        return
    ymin = min(values)
    ymax = max(values)
    pad = max(0.05, 0.12 * (ymax - ymin if ymax > ymin else 1))
    for ax in axes:
        ax.set_ylim(ymin - pad, ymax + pad)

## 5. Main t=-2 figure and citation source companion

In [5]:
companion_samples = [
    {
        "sample_key": "all_isolated_event_units",
        "title": "(a) All researchers",
        "selector": lambda frame: frame,
    },
    {
        "sample_key": "no_earlier_broad_service_evidence",
        "title": "(b) No earlier service",
        "selector": no_earlier_broad_service,
    },
]

def build_companion_figure(
    *,
    rows,
    value_col,
    ylabel,
    artifact_path,
    report_path,
    seed_base,
    plot_order,
    group_column,
    colors,
    offsets,
    markers,
    display_labels,
):
    companion_records = []
    fig, axes = plt.subplots(2, 1, figsize=(8.6, 6.1), sharex=True, sharey=True)

    for row_idx, (ax, sample) in enumerate(zip(axes, companion_samples)):
        sample_rows = sample["selector"](rows).copy()
        for group_idx, group in enumerate(plot_order):
            if group == "Aggregate":
                group_rows = sample_rows.copy()
            else:
                group_rows = sample_rows.loc[
                    sample_rows[group_column].eq(group)
                ].copy()
            n_units = group_rows["event_unit_id"].nunique()
            summary = bootstrap_summary(
                group_rows,
                value_col,
                seed=1234,
            )
            for _, summary_row in summary.iterrows():
                companion_records.append(
                    {
                        "scale": value_col,
                        "sample_key": sample["sample_key"],
                        "group": group,
                        "event_time": int(summary_row["event_time"]),
                        "n_event_units": int(n_units),
                        "mean": summary_row["mean"],
                        "ci_lower": summary_row["ci_lower"],
                        "ci_upper": summary_row["ci_upper"],
                    }
                )
            if summary.empty:
                continue

            ci_rows = summary.loc[summary["event_time"].ne(-2)].copy()
            x = ci_rows["event_time"].to_numpy(dtype=float) + offsets[group]
            yerr = np.vstack(
                [
                    ci_rows["mean"].to_numpy() - ci_rows["ci_lower"].to_numpy(),
                    ci_rows["ci_upper"].to_numpy() - ci_rows["mean"].to_numpy(),
                ]
            )
            ax.errorbar(
                x,
                ci_rows["mean"],
                yerr=yerr,
                fmt=markers[group],
                markersize=4.1,
                markerfacecolor=colors[group],
                markeredgecolor="black" if group == "Aggregate" else colors[group],
                markeredgewidth=0.45,
                color=colors[group],
                ecolor=colors[group],
                elinewidth=0.8,
                capsize=2.7,
                capthick=0.8,
                zorder=4 if group == "Aggregate" else 3,
                label=f"{display_labels[group]} ({n_units})",
            )
            tminus_two = summary.loc[summary["event_time"].eq(-2)]
            if not tminus_two.empty:
                ax.scatter(
                    tminus_two["event_time"].to_numpy(dtype=float) + offsets[group],
                    tminus_two["mean"],
                    marker=markers[group],
                    s=18,
                    color=colors[group],
                    edgecolors="black" if group == "Aggregate" else colors[group],
                    linewidths=0.45,
                    zorder=4 if group == "Aggregate" else 3,
                )

        ax.axhline(0, color="0.55", linewidth=0.8, linestyle="--", zorder=0)
        ax.axvline(0, color="0.82", linewidth=0.8, zorder=0)
        ax.set_title(sample["title"], loc="left", pad=16, fontsize=12)
        ax.set_ylabel(ylabel)
        ax.set_xlim(min(EVENT_TIMES) - 0.5, max(EVENT_TIMES) + 0.5)
        ax.grid(axis="y", color="0.88", linestyle=":", linewidth=0.55)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.65, 1.18),
            ncol=4,
            columnspacing=0.9,
            handletextpad=0.35,
            fontsize=8.5,
        )

    axes[-1].set_xlabel("Event time")
    axes[-1].set_xticks(EVENT_TIMES)
    fig.tight_layout(rect=[0, 0, 1, 0.98], h_pad=2.05)
    fig.savefig(artifact_path, bbox_inches="tight")
    fig.savefig(report_path, bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {artifact_path.relative_to(PROJECT)}")
    print(f"wrote {report_path.relative_to(PROJECT)}")
    return pd.DataFrame(companion_records)


main_rows = event_rows.loc[
    event_rows["plot_group"].isin(aggregate_groups)
].copy()
main_plot_order = ["Aggregate", "ICFP", "POPL", "PLDI 2021 onward"]
main_colors = {
    "Aggregate": "black",
    "ICFP": CONFERENCE_COLORS["ICFP"],
    "POPL": CONFERENCE_COLORS["POPL"],
    "PLDI 2021 onward": CONFERENCE_COLORS["PLDI"],
}
main_offsets = {
    "Aggregate": -0.18,
    "ICFP": -0.06,
    "POPL": 0.06,
    "PLDI 2021 onward": 0.18,
}
main_markers = {
    "Aggregate": "D",
    "ICFP": "o",
    "POPL": "s",
    "PLDI 2021 onward": "^",
}
main_display_labels = {
    "Aggregate": "Aggregate",
    "ICFP": "ICFP",
    "POPL": "POPL",
    "PLDI 2021 onward": "PLDI 2021 onward",
}

main_summary_log = build_companion_figure(
    rows=main_rows,
    value_col="delta_from_t_minus_2_log10_citations",
    ylabel=r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$",
    artifact_path=T_MINUS_MAIN_FIGURE_OUT,
    report_path=REPORT_T_MINUS_MAIN_FIGURE_OUT,
    seed_base=1234,
    plot_order=main_plot_order,
    group_column="plot_group",
    colors=main_colors,
    offsets=main_offsets,
    markers=main_markers,
    display_labels=main_display_labels,
)

main_summary_raw = build_companion_figure(
    rows=main_rows,
    value_col="delta_from_t_minus_2_raw_citations",
    ylabel=r"$\Delta$ citations from $t_{-2}$",
    artifact_path=T_MINUS_MAIN_RAW_FIGURE_OUT,
    report_path=REPORT_T_MINUS_MAIN_RAW_FIGURE_OUT,
    seed_base=1234,
    plot_order=main_plot_order,
    group_column="plot_group",
    colors=main_colors,
    offsets=main_offsets,
    markers=main_markers,
    display_labels=main_display_labels,
)

displayed_groups = ["ICFP", "POPL", "PLDI 2021 onward"]
source_shift_companion_rows = event_rows.loc[
    event_rows["plot_group"].isin(displayed_groups)
].copy()
source_shift_plot_order = ["Aggregate", "ICFP", "POPL", "PLDI 2021 onward"]
source_shift_colors = {
    "Aggregate": "black",
    "ICFP": CONFERENCE_COLORS["ICFP"],
    "POPL": CONFERENCE_COLORS["POPL"],
    "PLDI 2021 onward": CONFERENCE_COLORS["PLDI"],
}
source_shift_offsets = {
    "Aggregate": -0.18,
    "ICFP": -0.06,
    "POPL": 0.06,
    "PLDI 2021 onward": 0.18,
}
source_shift_markers = {
    "Aggregate": "D",
    "ICFP": "o",
    "POPL": "s",
    "PLDI 2021 onward": "^",
}
source_shift_display_labels = {
    "Aggregate": "Aggregate",
    "ICFP": "ICFP",
    "POPL": "POPL",
    "PLDI 2021 onward": "PLDI 2021 onward",
}

source_shift_companion_summary_log = build_companion_figure(
    rows=source_shift_companion_rows,
    value_col="delta_from_t_minus_2_log10_citations",
    ylabel=r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$",
    artifact_path=T_MINUS_SOURCE_SHIFT_COMPANION_FIGURE_OUT,
    report_path=REPORT_T_MINUS_SOURCE_SHIFT_COMPANION_FIGURE_OUT,
    seed_base=1234,
    plot_order=source_shift_plot_order,
    group_column="plot_group",
    colors=source_shift_colors,
    offsets=source_shift_offsets,
    markers=source_shift_markers,
    display_labels=source_shift_display_labels,
)

display(main_summary_log.head())
display(main_summary_raw.head())
display(source_shift_companion_summary_log.head())

wrote step_4_artifacts/figures_t_minus_event_study/t_minus_no_earlier_broad_service_main_text.pdf
wrote report_latex/figures/t_minus_no_earlier_broad_service_main_text.pdf
wrote step_4_artifacts/figures_t_minus_event_study/t_minus_no_earlier_broad_service_main_text_raw.pdf
wrote report_latex/figures/t_minus_no_earlier_broad_service_main_text_raw.pdf
wrote step_4_artifacts/figures_t_minus_event_study/t_minus_source_shift_companion.pdf
wrote report_latex/figures/t_minus_source_shift_companion.pdf


,scale,sample_key,group,event_time,n_event_units,mean,ci_lower,ci_upper
0,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,-2,349,0.000000,0.000000,0.000000
1,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,-1,349,0.011488,-0.029906,0.051216
2,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,0,349,0.034251,-0.006475,0.073640
3,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,1,349,0.004902,-0.037914,0.046709
4,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,2,349,0.008957,-0.033830,0.049684


,scale,sample_key,group,event_time,n_event_units,mean,ci_lower,ci_upper
0,delta_from_t_minus_2_raw_citations,all_isolated_event_units,Aggregate,-2,349,0.000000,0.000000,0.000000
1,delta_from_t_minus_2_raw_citations,all_isolated_event_units,Aggregate,-1,349,-0.297994,-0.991476,0.355372
2,delta_from_t_minus_2_raw_citations,all_isolated_event_units,Aggregate,0,349,-0.048711,-0.696347,0.587464
3,delta_from_t_minus_2_raw_citations,all_isolated_event_units,Aggregate,1,349,-0.332378,-1.071848,0.335745
4,delta_from_t_minus_2_raw_citations,all_isolated_event_units,Aggregate,2,349,-0.140401,-0.856877,0.524427


,scale,sample_key,group,event_time,n_event_units,mean,ci_lower,ci_upper
0,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,-2,349,0.000000,0.000000,0.000000
1,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,-1,349,0.011488,-0.029906,0.051216
2,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,0,349,0.034251,-0.006475,0.073640
3,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,1,349,0.004902,-0.037914,0.046709
4,delta_from_t_minus_2_log10_citations,all_isolated_event_units,Aggregate,2,349,0.008957,-0.033830,0.049684


## 6. Compare t=-2 and citation source shift filters

In [6]:
source_rows = pd.read_parquet(SOURCE_SHIFT_ROWS_PATH)
source_rows = source_rows.loc[source_rows["analysis_conference"].isin(displayed_groups)].copy()

def count_units(frame, no_broad=False):
    if no_broad:
        frame = frame.loc[
            frame["is_true_first_broad_service_in_observed_year_conf"].eq(True)
        ].copy()
    return int(frame["event_unit_id"].nunique())

tminus_display_rows = source_shift_companion_rows.copy()
tminus_single_service_rows = source_shift_companion_rows.loc[
    source_shift_companion_rows["n_pc_years_this_conference"].eq(1)
].copy()

comparison = pd.DataFrame(
    [
        {
            "check": "citation_source_shift",
            "filter": "balanced window + exactly one observed same-conference PC-service year",
            "included_conferences": "ICFP; POPL; PLDI 2021 onward",
            "all_event_units": count_units(source_rows, no_broad=False),
            "no_earlier_broad_service_event_units": count_units(source_rows, no_broad=True),
        },
        {
            "check": "t_minus_citation_count_companion",
            "filter": "balanced event window",
            "included_conferences": "ICFP; POPL; PLDI 2021 onward",
            "all_event_units": count_units(tminus_display_rows, no_broad=False),
            "no_earlier_broad_service_event_units": count_units(tminus_display_rows, no_broad=True),
        },
        {
            "check": "t_minus_citation_count_if_source_shift_filter_applied",
            "filter": "balanced event window + exactly one observed same-conference PC-service year",
            "included_conferences": "ICFP; POPL; PLDI 2021 onward",
            "all_event_units": count_units(tminus_single_service_rows, no_broad=False),
            "no_earlier_broad_service_event_units": count_units(tminus_single_service_rows, no_broad=True),
        },
    ]
)
comparison.to_csv(FILTER_COMPARISON_OUT, index=False)
display(comparison)
print(f"wrote {FILTER_COMPARISON_OUT.relative_to(PROJECT)}")

,check,filter,included_conferences,all_event_units,no_earlier_broad_service_event_units
0,citation_source_shift,balanced window + exactly one observed same-co...,ICFP; POPL; PLDI 2021 onward,263,87
1,t_minus_citation_count_companion,balanced event window,ICFP; POPL; PLDI 2021 onward,349,101
2,t_minus_citation_count_if_source_shift_filter_...,balanced event window + exactly one observed s...,ICFP; POPL; PLDI 2021 onward,263,87


wrote step_4_artifacts/summary_tables/tminus_source_filter_check.csv


## 7. Preview

In [7]:
notebook_root = Path("..") / ".."
for path in [
    T_MINUS_MAIN_FIGURE_OUT,
    T_MINUS_MAIN_RAW_FIGURE_OUT,
    T_MINUS_SOURCE_SHIFT_COMPANION_FIGURE_OUT,
    FILTER_COMPARISON_OUT,
]:
    display(Markdown(f"`{path.relative_to(PROJECT)}`"))

display(IFrame(src=str(notebook_root / T_MINUS_MAIN_FIGURE_OUT.relative_to(PROJECT)), width="100%", height=620))
display(IFrame(src=str(notebook_root / T_MINUS_MAIN_RAW_FIGURE_OUT.relative_to(PROJECT)), width="100%", height=620))
display(IFrame(src=str(notebook_root / T_MINUS_SOURCE_SHIFT_COMPANION_FIGURE_OUT.relative_to(PROJECT)), width="100%", height=620))

`step_4_artifacts/figures_t_minus_event_study/t_minus_no_earlier_broad_service_main_text.pdf`

`step_4_artifacts/figures_t_minus_event_study/t_minus_no_earlier_broad_service_main_text_raw.pdf`

`step_4_artifacts/figures_t_minus_event_study/t_minus_source_shift_companion.pdf`

`step_4_artifacts/summary_tables/tminus_source_filter_check.csv`